# 🔧 Model Configuration Manager for RAG Analysis

This notebook allows you to create, manage, and select between different **LLM model configurations** for use with tools like `ChatGPTAnalyzer`.

Each configuration includes:
- 🏷️ **Provider** (defaults to `Not Specified`; examples: `openai`, `gemini`, `anthropic`)
- 🤖 **Model name** (e.g., `gpt-4o`, `mistral-7b-instruct`, `llama3-70b-instruct`)
- 🌐 **Base URL** (e.g., `https://api.openai.com/v1`, `http://localhost:8080/v1`)
- 🔐 **API Key** (e.g., `sk-...` or `ollama`, `sk-noauth`)

---

## 🗃️ Configuration Storage

All configurations are stored in the following JSON file:

```
~/.secrets/model_configs.json
```

This file will look something like:

```json
{
  "openai": {
    "provider": "openai",
    "model": "gpt-4o",
    "base_url": "https://api.openai.com/v1",
    "api_key": "sk-..."
  },
  "mistral": {
    "provider": "Not Specified",
    "model": "mistral-7b-instruct",
    "base_url": "http://localhost:8080/v1",
    "api_key": "sk-noauth"
  },
  "_default": "openai"
}
```

You can **manually edit or copy** this file if needed to:
- Share configurations with others
- Maintain consistent setups across environments
- Set or change the default model (`"_default"` key)

---

## ✅ How to Use This Notebook

1. Run the notebook cells to:
   - Add or update a named model configuration
   - Optionally set it as the default

2. Use your configurations in tools like `ChatGPTAnalyzer`:

```python
from Open_AI_RAG_manager import ChatGPTAnalyzer

# Use specific model
analyzer = ChatGPTAnalyzer.from_config("mistral", yaml_content=my_yaml)

# Or just use the default model
analyzer = ChatGPTAnalyzer.from_config(yaml_content=my_yaml)
```

---

> 💡 Tip: You can define as many configurations as you like — great for testing different LLMs or backends like OpenAI, LM Studio, Ollama, and Hugging Face APIs.



In [ ]:
import json
from pathlib import Path
from getpass import getpass
from IPython.display import display, Markdown

CONFIG_FILE = Path.home() / ".secrets" / "model_configs.json"
CONFIG_FILE.parent.mkdir(exist_ok=True)

DEFAULT_KEY = "_default"
MODEL_PROVIDER_DEFAULT = "Not Specified"


def normalize_config(config):
    config = dict(config or {})
    config.setdefault("provider", MODEL_PROVIDER_DEFAULT)
    return config


def load_configs():
    if CONFIG_FILE.exists():
        with CONFIG_FILE.open() as f:
            return json.load(f)
    return {}


def save_configs(configs):
    with CONFIG_FILE.open("w") as f:
        json.dump(configs, f, indent=2)


def get_config_names(configs):
    return sorted([k for k in configs.keys() if k != DEFAULT_KEY])


def list_configs():
    configs = load_configs()
    names = get_config_names(configs)

    if not names:
        display(Markdown("❌ **No configurations found.**"))
        return

    default_name = configs.get(DEFAULT_KEY, "None")
    display(Markdown(f"### 📦 Saved Configurations (default: `{default_name}`)"))

    for name in names:
        cfg = normalize_config(configs[name])
        star = " ⭐" if name == default_name else ""
        display(Markdown(
            f"- `{name}`{star} → provider: `{cfg.get('provider', MODEL_PROVIDER_DEFAULT)}`, model: `{cfg.get('model','')}`, url: `{cfg.get('base_url','')}`"
        ))


def create_or_update_config():
    name = input("🔖 Enter configuration name (e.g., mistral): ").strip()
    if not name or name == DEFAULT_KEY:
        display(Markdown(f"❌ **Invalid name. `{DEFAULT_KEY}` is reserved.**"))
        return

    configs = load_configs()
    existing = normalize_config(configs.get(name, {}))

    provider = input(f"🏷️ Provider [{existing.get('provider', MODEL_PROVIDER_DEFAULT)}]: ").strip() or existing.get("provider", MODEL_PROVIDER_DEFAULT)
    model = input(f"🤖 Model name [{existing.get('model','')}]: ").strip() or existing.get("model", "")
    base_url = input(f"🌐 Base URL [{existing.get('base_url','')}]: ").strip() or existing.get("base_url", "")

    # Don't show existing key; allow keep or replace
    replace_key = input("🔐 Replace API Key? (y/N): ").strip().lower() == "y"
    if replace_key or "api_key" not in existing:
        api_key = getpass("🔐 API Key (or sk-noauth): ").strip()
    else:
        api_key = existing.get("api_key", "")

    configs[name] = {"provider": provider or MODEL_PROVIDER_DEFAULT, "model": model, "base_url": base_url, "api_key": api_key}

    make_default = input("⭐ Set this config as default? (y/N): ").strip().lower() == "y"
    if make_default:
        configs[DEFAULT_KEY] = name

    save_configs(configs)
    display(Markdown(f"✅ **Configuration `{name}` saved.**"))
    if make_default:
        display(Markdown("🌟 **Set as default configuration.**"))


def set_default_config():
    configs = load_configs()
    names = get_config_names(configs)
    if not names:
        display(Markdown("❌ **No configurations to set as default.**"))
        return

    list_configs()
    name = input("⭐ Enter config name to set as default: ").strip()
    if name not in configs or name == DEFAULT_KEY:
        display(Markdown(f"❌ **Config `{name}` not found.**"))
        return

    configs[DEFAULT_KEY] = name
    save_configs(configs)
    display(Markdown(f"🌟 **Default configuration set to `{name}`.**"))


def delete_config():
    configs = load_configs()
    names = get_config_names(configs)
    if not names:
        display(Markdown("❌ **No configurations to delete.**"))
        return

    list_configs()
    name = input("🗑️ Enter config name to delete: ").strip()
    if name not in configs or name == DEFAULT_KEY:
        display(Markdown(f"❌ **Config `{name}` not found.**"))
        return

    confirm = input(f"⚠️ Really delete `{name}`? (y/N): ").strip().lower() == "y"
    if not confirm:
        display(Markdown("✅ **Delete cancelled.**"))
        return

    configs.pop(name, None)

    # If it was default, pick a new default (first remaining) or clear default
    if configs.get(DEFAULT_KEY) == name:
        remaining = get_config_names(configs)
        if remaining:
            configs[DEFAULT_KEY] = remaining[0]
            display(Markdown(f"ℹ️ **Default deleted; new default is `{remaining[0]}`.**"))
        else:
            configs.pop(DEFAULT_KEY, None)
            display(Markdown("ℹ️ **Default deleted; no configs remain.**"))

    save_configs(configs)
    display(Markdown(f"🗑️ **Deleted `{name}`.**"))


def rename_config():
    configs = load_configs()
    names = get_config_names(configs)
    if not names:
        display(Markdown("❌ **No configurations to rename.**"))
        return

    list_configs()
    old = input("✏️ Enter config name to rename: ").strip()
    if old not in configs or old == DEFAULT_KEY:
        display(Markdown(f"❌ **Config `{old}` not found.**"))
        return

    new = input("✏️ Enter new name: ").strip()
    if not new or new == DEFAULT_KEY or new in configs:
        display(Markdown("❌ **Invalid new name (reserved, empty, or already exists).**"))
        return

    configs[new] = configs.pop(old)
    if configs.get(DEFAULT_KEY) == old:
        configs[DEFAULT_KEY] = new

    save_configs(configs)
    display(Markdown(f"✅ **Renamed `{old}` → `{new}`.**"))


def menu():
    while True:
        display(Markdown(
            "## 🔧 Model Config Manager\n"
            "1) List configs\n"
            "2) Create/Update config\n"
            "3) Set default config\n"
            "4) Delete config\n"
            "5) Rename config\n"
            "6) Exit"
        ))
        choice = input("➡️ Enter choice: ").strip()

        if choice == "1":
            list_configs()
        elif choice == "2":
            create_or_update_config()
        elif choice == "3":
            set_default_config()
        elif choice == "4":
            delete_config()
        elif choice == "5":
            rename_config()
        elif choice == "6":
            display(Markdown("👋 Done."))
            break
        else:
            display(Markdown("❌ **Invalid choice.**"))


# Run interactively
menu()


## 🔧 Model Config Manager
1) List configs
2) Create/Update config
3) Set default config
4) Delete config
5) Rename config
6) Exit

➡️ Enter choice:  3


### 📦 Saved Configurations (default: `gemini-flash`)

- `4o` → model: `gpt-4o`, url: `https://api.openai.com/v1`

- `Qwen3` → model: `qwen3-30b-a3b-thinking-2507`, url: `https://api.siemens.com/llm/v1`

...

### 📦 Saved Configurations (default: `Qwen3`)

- `4o` → model: `gpt-4o`, url: `https://api.openai.com/v1`

- `Qwen3` ⭐ → model: `qwen3-30b-a3b-thinking-2507`, url: `https://api.siemens.com/llm/v1`

- `gemini-flash` → model: `gemini-2.0-flash`, url: `https://generativelanguage.googleapis.com/v1beta/openai`

## 🔧 Model Config Manager
1) List configs
2) Create/Update config
3) Set default config
4) Delete config
5) Rename config
6) Exit